In [1]:
# pip install packages that are not in Pyodide
%pip install ipympl==0.9.3
%pip install seaborn==0.12.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [47]:
import time
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import seaborn as sns
from cycler import cycler
%matplotlib widget


# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

# Predicting Pressure in a Water Network using an MLP

 In this example, we demonstrate how a Multilayer Perceptron (MLP) can be used to estimate nodal pressures in a water distribution network, as illustrated in the simplified schematic below:

```{figure} https://files.mude.citg.tudelft.nl/WDSAsset1v1.png

---

---
Simplified scheme of a branched water distribution system
```








Pressure estimation is critical for ensuring stable and efficient operation of water networks. Traditionally, this is achieved using physically-based hydrodynamic models, which simulate fluid behavior across the system. However, such models can be computationally expensive, especially when used in tasks like optimization or real-time control in large networks.

To address this limitation, we use a data-driven alternative: a neural network trained on simulation results from the original physical model. This allows us to approximate the mapping from pipe diameters to nodal pressures, achieving high accuracy with significantly reduced computational cost.

As a case study, we use data from the BakRyan water distribution system (figure 2), which consists of 58 pipes and 35 nodes.

```{figure} https://files.mude.citg.tudelft.nl/BAK.png

---

---
Numerical results for the BakRyan water distribution system
```



### Mathematical Model


We can express the relationship between pipe diameters and nodal pressures using a function $\phi$ parameterized by the neural network weights $\mathbf{W}$:

$$
\mathbf{y} = \phi(\mathbf{x}, \mathbf{W})
$$

Where:

- **$\mathbf{y}$**: output data (nodal pressures, units: *mwc*)
- **$\phi$**: represents the Neural Network
- **$\mathbf{x}$**: input data (pipe diameters, units: *m*)
- **$\mathbf{W}$**: parameters of the MLP (unitless)



Having pairs of input-output data (**diameters** $\mathbf{x}$ and **pressures** $\mathbf{y}$) the goal is to find the parameters $\mathbf{W}$ that best fit the data.

The neural network we train is a fully connected MLP, as sketched below.


```{figure} https://files.mude.citg.tudelft.nl/ANN_image2.png

---

---
Artificial neural network representation, with a zoomed-in view of how a single neuron works. For this notebook, we have 58 input features (pipe diameters) and 35 outputs (nodal pressures).
```




## Load data
we load the data and below you can check the dimensions of the *features* (the pipe diameters) and the *targets* (the nodal pressures).

In [3]:
import os
from urllib.request import urlretrieve

def findfile(fname):
    if not os.path.isfile(fname):
        print(f"Downloading {fname}...")
        urlretrieve('http://files.mude.citg.tudelft.nl/'+fname, fname)

findfile('targets_BAK.csv')
findfile('features_BAK.csv')

features = np.loadtxt('features_BAK.csv', delimiter=",")
targets = np.loadtxt('targets_BAK.csv', delimiter=",")


This page contains interactive python element: click {fa}`rocket` --> {guilabel}`Live Code` in the top right corner to activate it.

In [12]:
print('Dimensions of features (X):', features.shape)
print('Dimensions of targets  (t):', targets.shape)

Dimensions of features (X): (10000, 58)
Dimensions of targets  (t): (10000, 35)


## Preprocessing


We use the *scikit-learn* to split and normalized data.

````{admonition} Coding neural nets with scikit-learn

As explained in the last chapter to work with MLPs we can use `scikit-learn`. Refer to the documentation of `scikit-learn` to find out more about the functios

[`sckit-learn`](https://scikit-learn.org/stable/api/index.html)
[`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
[`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

````
First we split the data and divide our dataset into three subsets:

- **Training set**: used to train the model.
- **Validation set**: used to tune hyperparameters and monitor for overfitting.
- **Test set**: used to assess final performance.

We aim for the following proportions:  
**80% training**, **10% validation**, **10% test**.

Since `train_test_split` operates on two splits at a time, we first split 80/20, then split the 20% again 50/50 into validation and test, meaning 10% each. It can be implemented in python code like this:


```python

from sklearn.model_selection import train_test_split

X_train, X_val_test, t_train, t_val_test = train_test_split(features, targets, test_size=0.20, random_state=42)
X_val, X_test, t_val, t_test = train_test_split(X_val_test, t_val_test, test_size=0.50, random_state=42)

```

We will now normalize the features and targets. Neural networks are sensitive to the **scale** of input features and target outputs. To ensure efficient training with gradient-based optimizers, we normalize all data to the range **[0, 1]** using the `MinMaxScaler` from `sklearn.preprocessing`.

Normalization should always be based **only on the training set** to avoid information leakage from validation or test data.


```python

from sklearn.preprocessing import MinMaxScaler

# Normalize feature variables
scaler_features = MinMaxScaler()
scaler_features.fit(X_train)

normalized_X_train = scaler_features.transform(X_train)
normalized_X_val = scaler_features.transform(X_val)
normalized_X_test = scaler_features.transform(X_test)

# Normalize target variables
scaler_targets = MinMaxScaler()
scaler_targets.fit(t_train)

normalized_t_train = scaler_targets.transform(t_train)
normalized_t_val = scaler_targets.transform(t_val)
normalized_t_test = scaler_targets.transform(t_test)

```






## Defining and training the model

To define an MLP we can use using scikit-learn's `MLPRegressor` class and specify the number of hidden layers and neurons. We can also choose different activation functions, such as `identity`, `logistic`, `tanh`, `relu`. For example:

```python

from sklearn.neural_network import MLPRegressor

model = MLPRegressor(
    hidden_layer_sizes=(..., ...),  # define number of neurons per hidden layer
    activation='function'            # activation function
)

```

### mini batches

The first step towards training a model is defining a function that transforms our training dataset into random mini-batches. This is a common practice used for training neural networks due to their computational efficiency and their ability to help the model generalize better. This practice generally leads to better computational efficiency, faster convergence and better generalization performance. 

The following figure illustrates both the way we split the original dataset and how we further split the training dataset into mini-batches. At every epoch the training dataset is shuffled and each mini-batch is considered **in isolation** by the network. The gradients coming from the single mini-batches are used to update the weights of the network (the randomness involved is why we say we are using **Stochastic** Gradient Descent).

```{figure} https://files.mude.citg.tudelft.nl/minibatching.png

---

---
Dataset splitting, mini-batching and the stochastic nature of MLP training.
```


Below is how you can implement this in python:

```python

def get_mini_batches(X, t, batch_size):
    """
    This function generates mini-batches from the given input data and labels.

    Parameters:
    X (numpy.ndarray): The features.
    t (numpy.ndarray): The targets corresponding to the input data.
    batchsize (int): The size of each mini-batch.

    Returns:
    list: A list of tuples where each tuple contains a mini-batch of the input data and the corresponding targets.
    """
    # Generate permutations
    perm = np.random.permutation(len(X))
    X_train_perm = X[perm]
    t_train_perm = t[perm]
    
    # Generate mini-batches
    X_batches = []
    t_batches = []
    for i in range(0, len(X_train_perm), batch_size):
        X_batches.append(X_train_perm[i:i+batch_size])
        t_batches.append(t_train_perm[i:i+batch_size])

    return list(zip(X_batches, t_batches))

```

In the training loop we define some hyperparamaters such as number of `epochs`, `batch size` and `learning rate`. To train the model we use a loss function, Mean Squared Error (MSE), which is defined as:

$$ MSE = \frac{1}{n} \sum_{i=1}^{n} (t_i - y_i)^2 $$

where $t_i$ is the actual target, $y_i$ is the predicted value, and $n$ is the number of samples.

Below you can see the structure of a training loop in python code:

```python
learning_rate = 0.001
n_epochs = 20
batch_size = 64

def train_model(model, normalized_X_train, normalized_t_train, normalized_X_val, normalized_t_val, n_epochs, batch_size, learning_rate):
    train_loss_list = [] 
    val_loss_list = []
    model.learning_rate_init = learning_rate
    
    for epoch in range(n_epochs):
        
        # Generate mini-batches
        mini_batches = get_mini_batches(normalized_X_train, normalized_t_train, batch_size)
        
        # Train model on mini-batches
        for X_batch, t_batch in mini_batches:
            model.partial_fit(X_batch, t_batch)

        # Compute loss on training and validation sets
        train_loss = mean_squared_error(normalized_t_train, model.predict(normalized_X_train))
        
        # Compute loss on validation set
        val_loss = mean_squared_error(normalized_t_val, model.predict(normalized_X_val))

        # Store loss values
        train_loss_list.append(train_loss)
        val_loss_list.append(val_loss)

        # Print training progress
        print(f"Epoch {epoch+1}/{n_epochs} - Train Loss: {train_loss_list[-1]:.4f} - Val Loss: {val_loss:.4f}")
        
    return train_loss_list, val_loss_list



Another metric that can be used is the $R^2$ score (coefficient of determination), that can be computed on both the training and validation set. It is a statistical metric that lays between 0 and 1 and indicates how well a regression model fits the data. It measures the proportion of the variance in the target variable that can be explained by the model’s features.

$R^2 = 1$ The model predicts perfectly

$R^2 = 0$ The model fails to capture any variability and it does not learn any relationship between the dependent and independent variable.



In scikit-learn, most regression models (including MLPRegressor) return the R² score when you call .score():

``` python

from sklearn.metrics import r2_score


train_r2 = model.score(X_train, y_train)   # R^2 on training set
val_r2   = model.score(X_val, y_val)       # R^2 on validation set
```



In the interactive plot below you can study the influence of the hyperparameters on the losses and the $R^2$ score. **Be aware that the required computations can take a few moments to run.** 

In [48]:

from ipywidgets import interact, IntSlider, Dropdown

# data prep
def prepare_data(features, targets):
    X_train, X_val_test, t_train, t_val_test = train_test_split(features, targets, test_size=0.20, random_state=42)
    X_val, X_test, t_val, t_test = train_test_split(X_val_test, t_val_test, test_size=0.50, random_state=42)
    
    scaler_X = MinMaxScaler().fit(X_train)
    scaler_t = MinMaxScaler().fit(t_train)
    
    return (
        scaler_X.transform(X_train),
        scaler_X.transform(X_val),
        scaler_t.transform(t_train),
        scaler_t.transform(t_val),
        scaler_X,
        scaler_t
    )

#Mini-batch 
def get_mini_batches(X, t, batch_size):
    perm = np.random.permutation(len(X))
    X_perm = X[perm]
    t_perm = t[perm]
    return [(X_perm[i:i+batch_size], t_perm[i:i+batch_size]) for i in range(0, len(X), batch_size)]

# Training loop
def train_model(model, X_train, t_train, X_val, t_val, n_epochs, batch_size):
    train_loss_list = []
    val_loss_list = []
    for epoch in range(n_epochs):
        mini_batches = get_mini_batches(X_train, t_train, batch_size)
        for X_batch, t_batch in mini_batches:
            model.partial_fit(X_batch, t_batch)
        train_loss_list.append(mean_squared_error(t_train, model.predict(X_train)))
        val_loss_list.append(mean_squared_error(t_val, model.predict(X_val)))
    return train_loss_list, val_loss_list

# Plotting 
def plot_losses(train_loss, val_loss):
    plt.figure(figsize=(8, 6))
    plt.plot(train_loss, label='Training loss', linewidth=2)
    plt.plot(val_loss, label='Validation loss', linewidth=2)
    plt.xlabel("Epochs", fontsize=14, fontweight='bold')
    plt.ylabel("Loss (MSE)", fontsize=14, fontweight='bold')
    plt.yscale("log")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.title("Loss Curves", fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.show()


# Interactive plot
def interactive_train_plot(features, targets):
    def run(num_layers=2, neurons=50, activation='tanh', n_epochs=20):
        X_train, X_val, t_train, t_val, _, _ = prepare_data(features, targets)
        hidden_layers = tuple([neurons] * num_layers)
        
        model = MLPRegressor(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            solver="sgd",
            learning_rate_init=0.001,
            max_iter=1,
            warm_start=True
        )
        
        train_loss, val_loss = train_model(model, X_train, t_train, X_val, t_val, n_epochs=n_epochs, batch_size=64)
        plot_losses(train_loss, val_loss)
    
    interact(
        run,
        num_layers=IntSlider(min=1, max=10, step=1, value=1, description="Layers"),
        neurons=IntSlider(min=10, max=100, step=5, value=10, description="Neurons"),
        n_epochs=IntSlider(min=10, max=100, step=5, value=30, description="Epochs"),
        activation=Dropdown(options=['identity', 'logistic', 'tanh', 'relu'], value='tanh', description='Activation')
    )

    
# Assumes your theme + color cycle are already set in a previous cell.
from ipywidgets import interact, IntSlider, Dropdown

from sklearn.metrics import mean_squared_error, r2_score


# --- Data preparation ---
def prepare_data(features, targets):
    X_train, X_val_test, t_train, t_val_test = train_test_split(
        features, targets, test_size=0.20, random_state=42, shuffle=True
    )
    X_val, _, t_val, _ = train_test_split(
        X_val_test, t_val_test, test_size=0.50, random_state=42, shuffle=True
    )
    scaler_X = MinMaxScaler().fit(X_train)
    scaler_t = MinMaxScaler().fit(t_train)

    X_train_n = scaler_X.transform(X_train)
    X_val_n   = scaler_X.transform(X_val)
    t_train_n = scaler_t.transform(t_train)
    t_val_n   = scaler_t.transform(t_val)
    return X_train_n, X_val_n, t_train_n, t_val_n

# --- Mini-batch generation ---
def get_mini_batches(X, t, batch_size):
    idx = np.random.permutation(len(X))
    Xp, tp = X[idx], t[idx]
    return [(Xp[i:i+batch_size], tp[i:i+batch_size]) for i in range(0, len(X), batch_size)]

# --- Training loop ---
def train_model(model, X_train, t_train, X_val, t_val, n_epochs, batch_size=64):
    train_loss_list, val_loss_list = [], []
    for _ in range(n_epochs):
        for Xb, tb in get_mini_batches(X_train, t_train, batch_size):
            model.partial_fit(Xb, tb)
        ytr = model.predict(X_train)
        yva = model.predict(X_val)
        train_loss_list.append(mean_squared_error(t_train, ytr))
        val_loss_list.append(mean_squared_error(t_val, yva))
    return train_loss_list, val_loss_list

# --- Plotting (uses your seaborn + rcParams theme) ---
def plot_losses(train_loss, val_loss, title_extra=""):
    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["#1f77b4", "#ff7f0e"])
    c_train = colors[0]
    c_val   = colors[1] if len(colors) > 1 else "#ff7f0e"

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(train_loss, label="Training loss", color=c_train, linewidth=2)
    ax.plot(val_loss,   label="Validation loss", color=c_val, linewidth=2)

    # Subtle markers for readability
    ax.scatter(range(len(train_loss)), train_loss, color=c_train, s=18, alpha=0.6)
    ax.scatter(range(len(val_loss)),   val_loss,   color=c_val,   s=18, alpha=0.6)

    ax.set_xlabel("Epochs", fontsize=14, fontweight="bold")
    ax.set_ylabel("Loss (MSE)", fontsize=14, fontweight="bold")
    ax.set_yscale("log")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_title(f"Loss Curves{title_extra}", fontsize=16, fontweight="bold")
    ax.legend(fontsize=12)
    fig.tight_layout()
    plt.show()

# --- Interactive widget ---
def interactive_train_plot(features, targets):
    def run(num_layers=2, neurons=50, activation="tanh", n_epochs=30):
        # Prepare data (normalized)
        X_train, X_val, t_train, t_val = prepare_data(features, targets)

        # Build model with chosen architecture
        hidden = tuple([neurons] * num_layers)
        model = MLPRegressor(
            hidden_layer_sizes=hidden,
            activation=activation,
            solver="sgd",
            learning_rate_init=1e-3,  # fixed
            max_iter=1,               # 1 epoch per partial_fit call
            warm_start=True,          # keep training across partial_fit calls
            random_state=0
        )

        # Train + collect losses (fixed batch_size=64)
        tr_loss, va_loss = train_model(model, X_train, t_train, X_val, t_val,
                                       n_epochs=int(n_epochs), batch_size=64)

        # Final R² for context
        r2_tr = r2_score(t_train, model.predict(X_train))
        r2_va = r2_score(t_val,   model.predict(X_val))

        title_extra = f"  |  R² train: {r2_tr:.3f} · R² val: {r2_va:.3f} "
        plot_losses(tr_loss, va_loss, title_extra=title_extra)

    interact(
        run,
        num_layers=IntSlider(min=1, max=10, step=1, value=2, description="Layers"),
        neurons=IntSlider(min=10, max=100, step=5, value=50, description="Neurons"),
        n_epochs=IntSlider(min=10, max=100, step=1, value=30, description="Epochs"),
        activation=Dropdown(options=["identity", "logistic", "tanh", "relu"],
                            value="tanh", description="Activation")
    )



interactive_train_plot(features, targets)


In [49]:
interactive_train_plot(features, targets)



interactive(children=(IntSlider(value=2, description='Layers', max=10, min=1), IntSlider(value=50, description…